# Rung 0 — the reliability of the assay, in plain language

**Task** `rung0-assay-reliability` · **Spec** [design.md](design.md) ·
**Decisions** [decisions.md](decisions.md) · **Review** [review.md](review.md) ·
**Audit** [audit.md](audit.md) · **Verification** [verification.md](verification.md)

Run this notebook top to bottom. Every number below is read from a committed table when you run
it, so what you see is what the artifacts say — not what anyone typed. To run it against another
run's artifacts, set the environment variable `RUNG0_TASK_DIR` to that directory.

---

## The question

How much of a measured drug response is signal rather than assay noise?

## How it is measured

Each cell line, drug and **dose** was screened on one or more plates. For every (line, drug, dose)
triple with two or more plates, sort the plate ids and assign them alternately to two groups,
average each group's per-gene log2 fold change, and correlate the two averaged profiles across
genes. Do that for every triple and take the mean. Spearman-Brown then lifts that half-data
correlation to the reliability of the full measurement.

Dose is part of the unit, never pooled. On this screen a given dose of a given drug on a given
line was mostly run once, so a split that pooled dose put different doses in the two halves for
99.7% of conditions and measured how well one dose agrees with another. The section "Why dose is
held fixed" at the end carries the counts.

Two gene sets, because they answer different questions:

- **All genes.** Most genes do not respond to most drugs, so this is largely a measure of how
  reproducibly the assay reports a flat profile.
- **Responders.** Only the genes that triple's **first** plate group called differentially
  expressed. Chosen from the first group and scored against the second, so no gene is picked
  using the half it is judged on.

And one decomposition: the screen publishes a standard error for every fold change, but that
error only sees cell sampling within a plate. Averaging each gene's variance across plates,
subtracting the average squared standard error, and flooring the difference at zero **once, after
averaging** says whether the noise a model actually meets is plate effects or cell sampling.

> ## Read this first
>
> **What is here.** The dose-fixed run: every replicated (line, drug, dose) triple, every gene,
> every drug, with its evidence tables and every figure the design declares. The cell below
> prints the run's own provenance from its parameter sidecar.
>
> **What is settled.** The split, the noise estimator, the effect-size control and the same-drug
> null were corrected on 2026-09-09 after the review of pull request #9
> ([review.md](review.md)); this run carries every correction. The ceilings a later rung divides
> by are declared at the **dose level** (`decisions.md`, 2026-09-11): the screen replicated its
> top dose far more than the other two and reproduces differently at each, so the mean over all
> triples is reported and not divided by, and each dose level is read against floors drawn from
> triples at its own dose.
>
> **Status.** Provisional until the fresh-reader re-audit ([audit.md](audit.md), "Re-audit")
> passes; nothing from this run is promoted before that.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

TASK = Path(os.environ.get("RUNG0_TASK_DIR", "docs/tasks/rung0-assay-reliability"))
if not TASK.exists():  # allow running from inside the task folder
    TASK = Path(".")
FIG = TASK / "figures"
KEYS = ["patient", "drug", "dose"]


def table(name, **kw):
    """Read a committed table, or say plainly that the run has not happened yet.

    keep_default_na=False matters here and is not a detail: one cell line's DepMap identifier
    is missing upstream and appears as the literal string "NA". Pandas reads that as a missing
    value by default, which would silently drop a real line from every count this notebook
    prints.
    """
    p = TASK / name
    if not p.exists():
        print(f"[not present yet] {p}")
        return None
    kw.setdefault("keep_default_na", False)
    kw.setdefault("na_values", [""])
    return pd.read_csv(p, **kw)


def show(name, caption=""):
    p = FIG / name
    if not p.exists():
        print(f"[figure not present yet] {p}")
        return
    if caption:
        print(caption)
    display(Image(filename=str(p)))


summary = table("rung0_reliability.csv")
S = summary.iloc[0].to_dict() if summary is not None else {}


def g(k, default=float("nan")):
    return S.get(k, default)


print("run present:", summary is not None)

sidecar = TASK / "rung0_reliability.params.json"
if sidecar.exists():
    params = json.loads(sidecar.read_text())
    print("\n### Provenance of what you are reading")
    for key in ("git_sha", "slurm_job_id", "split_rule", "dose_handling", "weighting"):
        print(f"  {key:14s} {params.get(key)}")
    print(f"  {'tranche':14s} tahoe100m-pseudobulk-de.v1, 1,026 shards, content-hash pinned")

## The hypotheses, stated before the run

From `design.md`, "Expected result", carried over verbatim. Each is answered below with the
number that settles it.

1. The correlations between plates of differential expression over all genes will be low, since
   most genes are not affected by the drugs.
2. Correlations between genes that were significantly different (responders) in group 1 will be
   higher than the correlations over all genes.
3. The noise will be higher in cell line - drug combinations with lower correlations over either
   all genes or the differentially expressed genes.
4. The aggregate noise will be higher in the differentially expressed genes (responders) than
   over all genes.

In [ ]:
if summary is not None:
    rows = []
    for fam, label in (("all", "all genes"), ("responder", "responders")):
        rows.append(
            {
                "gene set": label,
                "triples scored": int(g(f"{fam}_n_pairs")),
                "split-half r (mean)": g(f"{fam}_splithalf_mean_r"),
                "median": g(f"{fam}_splithalf_median_r"),
                "quartiles": (f"{g(f'{fam}_splithalf_q1_r')} - {g(f'{fam}_splithalf_q3_r')}"),
                "Spearman-Brown ceiling": g(f"{fam}_spearman_brown_full"),
                "SB, equal-half triples": g(f"{fam}_spearman_brown_full_even_plates"),
                "triples equal-half": int(g(f"{fam}_n_pairs_even")),
                "vs different-drug floor": g(f"{fam}_null_diff_drug_mean_r"),
                "vs same-drug, same-dose floor": g(f"{fam}_null_same_drug_mean_r"),
                "p vs different-drug": g(f"{fam}_p_vs_null"),
                "p vs same-drug": g(f"{fam}_p_vs_same_drug"),
                "MDE (80% power)": g(f"{fam}_mde_80_vs_diff_drug"),
            }
        )
    display(pd.DataFrame(rows).set_index("gene set").T)

In [ ]:
if summary is not None:
    a, r = g("all_splithalf_mean_r"), g("responder_splithalf_mean_r")
    print(
        f"Hypothesis 2 -- responders above all genes: "
        f"{'HELD' if r > a else 'DID NOT HOLD'} ({r:.3f} vs {a:.3f}, difference {r - a:+.3f})"
    )
    for fam, label in (("all", "all genes"), ("responder", "responders")):
        clears = g(f"{fam}_splithalf_mean_r") > max(
            g(f"{fam}_null_diff_drug_mean_r"), g(f"{fam}_null_same_drug_mean_r")
        )
        p_diff, p_same = g(f"{fam}_p_vs_null"), g(f"{fam}_p_vs_same_drug")
        print(
            f"{label}: clears both floors: {clears}; "
            f"p = {p_diff} (different-drug), {p_same} (same-drug, same-dose)"
        )

---

## Step 1 — build: what the screen actually contains

Before any statistic, what is in the pool. The unit is the (line, drug, dose) triple; the
composition panels are the denominator every later count is a subset of; the fold-change panel
puts the real screen beside a synthetic pool with a planted answer, so the control is visibly the
same shape as the data it stands in for rather than assumed to be.

In [ ]:
pool = table("rung0_pool_description.csv")
if pool is not None:
    rep = pool[pool["n_plates"] >= 2]
    print(f"(line, drug, dose) triples in the pool: {len(pool):,}")
    print(
        f"cell lines: {pool['patient'].nunique()}   drugs: {pool['drug'].nunique()}   "
        f"dose levels: {sorted(pool['dose'].unique())}"
    )
    print(
        f"triples with two or more plates (replicated, so splittable): {len(rep):,} "
        f"across {rep['drug'].nunique()} drugs and {rep['patient'].nunique()} lines"
    )
    print(
        f"replicated triples whose plates split into equal halves: "
        f"{int(rep['n_plates_even'].sum()):,}"
    )
    print("\nplates per replicated triple:")
    display(rep["n_plates"].value_counts().sort_index().rename("triples").to_frame().T)
    print("replicated triples per dose level:")
    display(rep["dose"].value_counts().sort_index().rename("triples").to_frame().T)
show("01_build.png")

## Step 2 — split: one triple becomes two half-profiles

Within each triple the distinct plate ids are sorted as text and assigned alternately — first to group 0,
second to group 1, third to group 0 — one fixed, seedless split, written out as a table
(`rung0_split_assignment.csv`) rather than evaluated inside the aggregations. Every replicated
triple therefore has a plate on each side, and "equal halves" means exactly "an even plate
count". Most replicated triples have two plates, so the split is one plate against one, the case
Spearman-Brown's correction is exactly right for; the corrected value is still reported again
over the equal-halves subset so the assumption is read off the data.

In [ ]:
per_pair = table("rung0_per_pair_r.csv")
split = table("rung0_split_assignment.csv")
if per_pair is not None:
    print(f"triples with a scoreable all-gene correlation: {per_pair['r'].notna().sum():,}")
    if "r_responder" in per_pair:
        print(
            f"triples with a scoreable responder correlation: "
            f"{per_pair['r_responder'].notna().sum():,}"
        )
    print(f"median genes scored per triple: {per_pair['n_genes_scored'].median():,.0f}")
if split is not None:
    per_half = split.groupby(KEYS)["half"].agg(
        half0=lambda h: int((h == 0).sum()), half1=lambda h: int((h == 1).sum())
    )
    both_sides = int(((per_half["half0"] > 0) & (per_half["half1"] > 0)).sum())
    print(
        f"(triple, plate) rows in the split assignment: {len(split):,}; "
        f"triples with a plate on each side: {both_sides:,}"
    )
show("02_split.png")

## Step 3 — select: which genes count as responders

A responder is a gene the triple's **first** plate group called differentially expressed.
Selection reads that group alone and the correlation is still first group against second.

Panel (c) is why. Both bars come from a pool with no signal at all. Selecting on the pooled data
— the natural mistake of calling differential expression on every plate at once and then
correlating the halves — keeps the genes whose noise agreed, and reads as reproducibility that
nothing generated. The gap between the bars is the bias the one-sided rule avoids.

Panel (d) is how far the two groups agree on which genes responded. It is a diagnostic and never
an input: keeping the genes both groups called *is* the pooled rule panel (c) measures.

In [ ]:
leak = table("rung0_leakage_control.csv")
if leak is not None:
    display(leak)
    one = float(leak.loc[leak["rule"] == "one-sided", "mean_r"].iloc[0])
    pooled = float(leak.loc[leak["rule"] == "pooled", "mean_r"].iloc[0])
    print(
        f"\nOn signal-free data the shipped one-sided rule reads {one:+.3f}; "
        f"selecting on the pooled data reads {pooled:+.3f}."
    )
    print(f"The inflation avoided is {pooled - one:+.3f}.")

ov = table("rung0_responder_overlap.csv")
if ov is not None:
    print(f"\nresponders per triple, first group: median {ov['n_first'].median():,.0f}")
    print(f"Jaccard overlap of the two groups' responder sets: median {ov['jaccard'].median():.3f}")
if per_pair is not None and "n_responders" in per_pair:
    med = per_pair["n_responders"].median()
    print(f"responders actually scored per triple: median {med:,.0f}")
show("03_select.png")

### A property of the selection rule, stated plainly

The rule is "significant in **at least one** of the first group's rows". Under the null a gene is
therefore admitted with probability `1 - (1 - alpha)^k` over `k` rows, not `alpha`. With dose held
fixed the first group is usually one plate, so `k` is usually one and the rate is close to the
nominal five percent; where the first group holds two plates it is about ten.

This does not bias the responder reliability: the mismatched-pair nulls apply the same rule to
the same group, so the comparison stays like for like. Its effect is directional and works
*against* the hypothesis — a larger responder set under the null dilutes the responder statistic
toward the all-gene one. Whatever separation is reported above is therefore conservative.

## Step 4 — score: the two reliabilities

The scatters are individual triples, drawn twice: over all genes, then over that triple's
responders. Each panel's correlation is recomputed from the points plotted and written to a
companion file, so the printed number is checkable rather than asserted.

The histograms beneath are every triple's correlation, with the control pools underneath on
shared axes — a pool with a planted reliability, and a pool with none.

In [ ]:
idx = table("rung0_example_pair_index.csv")
if idx is not None:
    display(idx)
show("04_score.png")

## Step 5 — decompose: what kind of noise the ceiling is made of

The screen's own `lfcSE` is the standard error of one plate's treated-versus-control contrast: it
sees cell sampling and cannot see plate-to-plate variation. For each gene, dose and condition
with at least two plates, the variance of the fold change across plates has expectation
`sigma^2_plate + mean(lfcSE^2)`.

The estimator is **pooled**: the mean over gene-conditions of the variance across plates, minus
the mean squared standard error, floored at zero once. The order matters. With two plates each
gene's variance has one degree of freedom, so the per-gene difference lands either side of zero
even with no plate effect; flooring each gene first would report a share of about 0.15 from
nothing (a third of genes positive). Averaging first is unbiased, because expectation is linear,
and the floor then constrains one well-estimated number. Panel (a) shows the per-gene spread with
the pooled value marked, beside a control pool that plants a share of one half at two plates.

If the plate component dominates, the published standard errors understate assay noise and the
split-half is the only honest ceiling. If it is near zero, the assay's noise is cell sampling —
and a later rung could in principle read against a ceiling derived from `lfcSE` on material with
no replicate plates at all, which is the only route to a ceiling for unreplicated samples.

In [ ]:
noise = table("rung0_noise_decomposition.csv")
if noise is not None:
    n = noise.iloc[0]
    display(noise.T.rename(columns={0: "value"}))
    share = float(n["between_plate_fraction_pooled"])
    print(
        f"\nBetween-plate share of the fold-change variance, pooled over "
        f"{int(n['n_gene_conditions']):,} gene-conditions: {share:.3f}"
    )
    print(
        f"  the same share pooled within the responders: "
        f"{float(n['between_plate_fraction_pooled_responders']):.3f}; "
        f"within the non-responders: {float(n['between_plate_fraction_pooled_nonresponders']):.3f}"
    )
    print(
        f"  mean over {int(n['n_conditions_decomposed']):,} conditions of each condition's "
        f"own share: {float(n['between_plate_fraction_pooled_over_conditions']):.3f}"
    )
    print(
        f"  conditions where plate effects dominate (share > 0.5): "
        f"{float(n['frac_conditions_plate_dominated']):.1%}"
    )
    verdict = (
        "plate effects dominate, so the published standard errors understate assay noise "
        "and the split-half is the only honest ceiling"
        if share > 0.5
        else "cell sampling dominates, so the published standard errors and the split-half "
        "describe substantially the same noise"
    )
    print(f"\nReading: {verdict}.")

# The stratified view: each stratum's share pooled the same way, from the committed sample.
strata = table("rung0_noise_strata.csv")
if strata is not None:

    def pooled_by(df, key):
        out = {}
        for level, part in df.groupby(key):
            w = part["n"].to_numpy(dtype=float)
            var = float((part["var_lfc_mean"].to_numpy(dtype=float) * w).sum() / w.sum())
            se2 = float((part["mean_se2_mean"].to_numpy(dtype=float) * w).sum() / w.sum())
            out[int(level)] = round(max(var - se2, 0.0) / var, 4) if var > 0 else float("nan")
        return pd.Series(out, name="pooled between-plate share")

    print("\nby expression quartile (1 = lowest baseMean):")
    display(pooled_by(strata, "expression_quartile").to_frame().T)
    print("by response-size quartile (1 = smallest mean absolute log2 fold change):")
    display(pooled_by(strata, "response_quartile").to_frame().T)
show("05_decompose.png")

### Hypotheses 3 and 4 — noise per condition

Both are statements about individual cell line, drug and dose combinations rather than about the
screen's average, so neither follows from the single number above.

**Hypothesis 3** says noise should be higher where the correlations are lower, over either gene
set. That is the tercile control seen from the other side, and it should hold for the same
reason: a condition whose replicate variance is mostly noise has less signal left to reproduce.

**Hypothesis 4** says the responder genes should carry more noise than genes overall. Responders
are selected for having moved, and a gene that moved has more variance to decompose — so this is
close to a check that selection did what it says, and it would be surprising if it failed.

In [ ]:
by_cond = table("rung0_noise_by_condition.csv")
if by_cond is not None and per_pair is not None:
    j = per_pair.merge(by_cond, on=KEYS, how="inner")
    for column, label in (("r", "all genes"), ("r_responder", "responders")):
        if column not in j:
            continue
        ok = j[column].notna() & j["var_lfc_mean"].notna()
        if int(ok.sum()) > 2:
            # Spearman is Pearson on ranks; written that way so the notebook needs no scipy
            rho = float(j.loc[ok, column].rank().corr(j.loc[ok, "var_lfc_mean"].rank()))
            print(
                f"Hypothesis 3 ({label}) -- noise higher where correlations are lower: "
                f"{'HELD' if rho < 0 else 'DID NOT HOLD'}"
            )
            print(
                f"  Spearman correlation between a triple's split-half r and its mean "
                f"replicate variance: {rho:+.3f} over {int(ok.sum()):,} triples"
            )
    print("  (a negative value is the hypothesis: more noise, less reproducibility)")

    # Hypothesis 4 is stated in the aggregate: "the aggregate noise will be higher in the
    # differentially expressed genes than over all genes". That is the variance across plates
    # pooled over gene-conditions, read from the decomposition table; the per-triple view is
    # given beside it because the two disagree here, and the disagreement is itself a finding.
    if noise is not None:
        n = noise.iloc[0]
        v_all, v_resp = float(n["var_lfc_mean"]), float(n["var_lfc_mean_responders"])
        print(
            f"\nHypothesis 4 -- aggregate noise higher in the responders than over all genes: "
            f"{'HELD' if v_resp > v_all else 'DID NOT HOLD'}"
        )
        print(
            f"  variance across plates pooled over gene-conditions: responders {v_resp:.4f} "
            f"vs all genes {v_all:.4f} "
            f"(non-responders {float(n['var_lfc_mean_nonresponders']):.4f})"
        )
    resp = j["var_lfc_mean_responders"]
    non = j["var_lfc_mean_nonresponders"]
    both = resp.notna() & non.notna()
    if int(both.sum()) > 0:
        frac = float((resp[both] > non[both]).mean())
        print(
            f"  triple by triple the picture differs: responders are the noisier set in "
            f"{frac:.1%} of {int(both.sum()):,} triples (per-triple means {resp[both].mean():.4f} "
            f"vs {non[both].mean():.4f}). Responders are chosen on plate 0's significance, which "
            f"widens the plate-0-against-plate-1 difference by construction, so the aggregate is "
            f"read with that in mind and neither number is cited as a plate share."
        )

## Step 6 — null: what the reliabilities are read against

A split-half correlation has a floor above zero, because genes share structure whether or not two
profiles come from the same perturbation. Three floors are built by pairing one triple's first
group with a *different* triple's second group:

- **any pair** — reported for continuity only.
- **different drug and line** — the generic-structure floor a ceiling must clear to be a ceiling.
- **same drug at the same dose, different line** — the stricter, line-specificity floor: two
  lines given one drug at one dose already share that drug's generic response, so clearing this
  says the response is specific to the line rather than to the compound. Dose is held fixed in
  the floor for the same reason it is held fixed in the condition.

A mismatched draw for the responder statistic uses the *first* triple's responder genes — the
same selection rule as a matched pair, so the null answers the same question.

In [ ]:
nulls = table("rung0_null_draws.csv")
if nulls is not None:
    display(nulls.groupby(["gene_set", "stratum"])["r"].agg(["count", "mean", "median"]).round(4))
show("06_null.png")

### Which genes carry reproducible signal

The same measurement transposed: instead of correlating two halves of one triple across genes,
correlate two halves of one gene across triples. It says which genes report a reproducible
response at all — the evidence base a later rung would need if it ever wanted to restrict to a
gene panel, which this rung deliberately does not do.

In [ ]:
per_gene = table("rung0_per_gene_reliability.csv")
if per_gene is not None:
    scored = per_gene["r"].notna()
    print(f"genes with a scoreable across-triple correlation: {int(scored.sum()):,}")
    print(
        f"median {per_gene['r'].median():.3f}; {int((per_gene['r'] > 0.2).sum()):,} genes above 0.2"
    )
show("09_per_gene_reliability.png")

In [ ]:
for gene_set, fname, fig in (
    ("all genes", "rung0_permutation_summary.csv", "11_permutation_vs_bootstrap.png"),
    (
        "responders",
        "rung0_permutation_summary_responder.csv",
        "11_permutation_vs_bootstrap_responder.png",
    ),
):
    perm = table(fname)
    if perm is not None:
        row = perm.iloc[0]
        print(f"--- permutation check, {gene_set}: {int(row['n_perm'])} permutations ---")
        print(
            f"  observed mean {row['observed_mean']:.4f} against the permutation null "
            f"{row['perm_mean_mean_diff_drug']:.4f} (different drug) and "
            f"{row['perm_mean_mean_same_drug']:.4f} (same drug): "
            f"{row['z_permutation']:.0f} null sds above; exact p {row['p_exact_diff_drug']}"
        )
        print(f"  design effect, different-drug stratum: {row['design_effect_diff_drug']:.2f}")
        display(perm.T.rename(columns={0: "value"}))
        show(fig)

The bootstrap that produces the p-values above treats the mismatched draws as independent, and
they are not: every draw reuses the same half-profiles. The permutation check carries that
dependence by construction — it permutes the pairing so no triple meets its own partner — and
reports the **design effect**, the ratio of the true sampling variance of the mean to the variance
an independent pool would have. A design effect near 1 means the bootstrap was not misled; a large
one means its p-values are optimistic by that factor in variance; below 1, the bootstrap was
conservative.

Read the **different-drug** design effect (`design_effect_diff_drug`), the stratum the p-value is
read against. The pooled `design_effect` divides by the variance of an any-pair pool of 100 draws
that mixes same-drug and different-drug pairs, and that variance swings with how many same-drug
pairs a draw catches; it read 0.78 at 500 permutations and 1.26 at 100 for the same data
(`decisions.md`, 2026-09-11). With 100 permutations the smallest exact p-value is 0.0099, so a p
at that value means "beyond every permutation", not a p of 0.01.

## Step 7 — dose: where the replicated triples sit, and how they reproduce

Holding dose fixed made the unit a triple, and the screen replicated its doses unevenly. Every
candidate for "the" ceiling is an aggregate of the per-triple table: each dose level on its own;
all triples with equal weight (what the summary row reports); each (line, drug) pair weighted
once, its triples averaged first. They are shown together so the choice is read off committed
numbers. Each dot in the figure is one triple's correlation at its cell line, coloured by dose;
beside it, the distribution at each dose level.

In [ ]:
dose_strata = table("rung0_dose_strata.csv")
if dose_strata is not None:
    for gene_set in ("all", "responder"):
        part = dose_strata[dose_strata["gene_set"] == gene_set]
        if len(part):
            print(f"--- {gene_set} genes ---")
            display(part.drop(columns=["gene_set"]).set_index(["weighting", "dose"]))
    ceilings = dose_strata[
        (dose_strata["weighting"] == "per_triple") & (dose_strata["dose"] != "all")
    ]
    if "null_same_drug_mean_r" in ceilings:
        print("the declared ceilings, each against floors drawn at its own dose:")
        for _, c in ceilings.iterrows():
            clears = (
                c["splithalf_mean_r"] > c["null_diff_drug_mean_r"]
                and c["splithalf_mean_r"] > c["null_same_drug_mean_r"]
                and c["p_vs_null"] < 0.05
                and c["p_vs_same_drug"] < 0.05
            )
            print(
                f"  {c['gene_set']:<9} {c['dose']:>5} uM: r {c['splithalf_mean_r']:.3f} "
                f"(Spearman-Brown {c['spearman_brown_full']:.3f}) over {int(c['n_pairs']):,}; "
                f"floors {c['null_diff_drug_mean_r']:.3f} / {c['null_same_drug_mean_r']:.3f}; "
                f"p {c['p_vs_null']} / {c['p_vs_same_drug']} -> "
                f"{'CLEARS its floors' if clears else 'does NOT clear its floors'}"
            )
    top = dose_strata[
        (dose_strata["gene_set"] == "all")
        & (dose_strata["weighting"] == "per_triple")
        & (dose_strata["dose"] != "all")
    ]
    if len(top):
        by_dose = top.set_index("dose")["n_pairs"]
        share = float(by_dose.max() / by_dose.sum())
        print(
            f"\nthe most replicated dose level ({by_dose.idxmax()}) carries {share:.0%} of the "
            f"scored triples, so the equal-weight mean over triples is mostly a statement about it"
        )
show("10_dose.png")

**The declaration** (`decisions.md`, 2026-09-11). The ceilings are the **dose-level rows**:
each dose level's mean over its triples, read against mismatched-pair floors drawn from triples
at that dose alone, with its own p-values and minimum detectable effects. The mean over all
triples is a blend weighted by where the screen happened to replicate, and it is reported, not
divided by. A later rung reads against the ceiling at the dose it scores, restricted to its own
triples; a dose level that does not clear its floors has no ceiling to read against.

## The controls, and what they establish

Every measurement step ships a positive control that plants a known answer and requires the
shipped code to recover it, and a negative control that feeds signal-free data and requires null.
These run in continuous integration, not only here.

The effect-size control below ranks triples into thirds by the response size ONE half measured
and averages the correlation of the pair within each third, then swaps halves. Ranking by one
half alone is what makes it a control: under no signal the other half is independent of the
ranking and every third sits at zero, whereas ranking by the two halves' sum selects triples whose
halves happened to agree and pure noise rises. Both rankings must rise.

In [ ]:
terc = table("rung0_effect_terciles.csv")
if terc is not None:
    display(terc)
    for ranked_by in ("half0", "half1"):
        part = terc[terc["ranked_by"] == ranked_by].sort_values("tercile")
        rising = part["mean_r"].is_monotonic_increasing and len(part) == 3
        print(
            f"Empirical control, ranked by {ranked_by} -- reproducibility rises with "
            f"response size: {'HELD' if rising else 'DID NOT HOLD'}"
        )
    print("An assay that cannot find more reproducibility where there is more signal is broken.")
show("07_terciles.png")

In [ ]:
mde = table("rung0_mde_curve.csv")
if mde is not None and summary is not None:
    obs = mde[mde["observed"]]
    display(obs)
    print("\nThe smallest effect this screen could have detected at 80% power, at its own")
    print("triple count. Reported so a null result cannot be confused with an underpowered")
    print("one -- the distinction the organoid rung will turn on, with a tenth of the conditions.")
show("08_power.png")

## Conclusions

Read the four hypotheses against the numbers above. Where one did not hold, that is the finding,
not a defect to explain away.

In [ ]:
if summary is not None:
    a, r = g("all_splithalf_mean_r"), g("responder_splithalf_mean_r")
    sb_a, sb_r = g("all_spearman_brown_full"), g("responder_spearman_brown_full")
    print("1. All-gene correlations are low:", f"mean r = {a:.3f} (ceiling {sb_a:.3f})")
    print(
        "2. Responders exceed all genes:",
        f"{'HELD' if r > a else 'DID NOT HOLD'} -- {r:.3f} vs {a:.3f} (ceiling {sb_r:.3f})",
    )
    print("3. and 4. Noise per condition: the verdicts printed above, under the heading")
    print("   'Hypotheses 3 and 4'. A hypothesis that did not hold is the finding, not a")
    print("   defect to soften -- the design stated all four before the run for that reason.")
    print()
    print("What a later rung divides by: the ceiling at the dose it scores, from the dose")
    print("section above, each against floors drawn at its own dose. The mean over all triples")
    print(f"(all genes {sb_a:.3f}, responders {sb_r:.3f} corrected) is reported, not divided by.")

## What this rung does not establish

- Sensitivity of either number to the **choice of split**. One fixed, alternating split is used.
- Any **restriction** of these ceilings to a later rung's genes, drugs or dose levels. Each
  later rung declares and computes its own, before it scores anything.

## Scripts this task touched

`scripts/delta_reproducibility.py` (assign, slice, combine: build, split, select, score,
decompose, null, dose strata, exports, figures) · `scripts/permutation_null.py` (the permutation
check) · `src/fmharness/statistics.py` (significance, power, Spearman-Brown) ·
`src/fmharness/figures.py` · `src/fmharness/synthetic.py` (the planted control pools) ·
`scripts/promote_result.py` (provenance from the run's own sidecar) · `scripts/verify_rung0.py`
and [verify.ipynb](verify.ipynb) (the claim-by-claim recomputation) ·
`scripts/alpine/rung0_assign.sbatch`, `rung0_slice.sbatch`, `rung0_combine.sbatch`,
`permutation_null.sbatch`, `submit_rung0_chain.sh`.

Controls and known-answer tests: `tests/test_rung0_controls.py`, `tests/test_rung0_figures.py`,
`tests/test_statistics_known_answers.py`, `tests/test_permutation_null.py`,
`tests/test_promote_result.py`, `tests/test_verify_rung0.py`.

---

## Why dose is held fixed

The first design pooled dose: a condition was one cell line and one drug, its plates were split
in two, and each half averaged over the screen's three doses. That reading assumed every dose was
replicated across a condition's plates. Counted on the cluster over the key columns alone (jobs
31996238, 31996294, 31996456), it is not:

| | |
|---|---|
| (line, drug, dose) triples | 56,827 |
| of those, on a **single plate** | **49,186 (86.6%)** |
| with 2+ plates | 7,641 (13.4%) |
| pooled-dose conditions whose two halves carried the **same** doses | **50 of 18,350 (0.3%)** |
| pooled-dose conditions whose halves carried **different** doses | **18,300 (99.7%)** |

A given dose of a given drug on a given line was mostly run once, so splitting a condition's
plates almost always split its doses, and the pooled-dose correlation (0.118 over all genes,
promoted as provisional on 2026-09-02 and superseded) measured how well one dose agrees with a
different one. Holding dose fixed costs base — 7,641 replicated triples against 18,350 pooled
conditions — and buys the quantity the rung is named for. The base is not evenly spread: 71% of
it is the top dose, which is why the dose section above exists.

The correction was found by a memory failure, not by reading the design: the noise decomposition
kept exhausting memory, and the slice counts it printed implied far fewer surviving gene-groups
than expected, which only makes sense if dose-conditions are mostly unreplicated. A drift audit
checks that the code does what the design says; it cannot catch a design that says something the
data will not support. Only running it and looking at the shape of what comes back does that.

## Working notes

**Measure where the cost is before asking for more of anything.** The table is 4,089,820,780
rows over 1,026 shards, 83 GB, and reading it is the entire cost of this measurement. Seven
cluster jobs on the first design failed on memory or wall clock before that was measured;
each fix addressed what the error message pointed at. What worked, and is kept:

- **Cheap probes before expensive jobs.** Every question that changed the design was answerable
  over the key columns alone in about fifteen minutes.
- **One scan per slice of the genes, one group-by serving both the reliabilities and the noise
  decomposition, run as a job array.** Gene is in every group key, so slices concatenate and
  sums add exactly; sixteen tasks on sixteen nodes make the wall clock one scan instead of
  sixteen in a row, and a finished slice is never repeated.
- **Per-slice caching, with a completion record written last.** A task killed mid-write leaves
  no half-slice a combine could mistake for a whole one; a failed index is resubmitted alone.
- **Twice the engine's memory limit for the job.** The parquet readers and the group keys sit
  outside DuckDB's accounting; the first assign job at 64 GB with the engine at 48 GB was killed
  by the cgroup at 67 GB.
- **Figures drawn from committed tables.** Written for reviewability; it is also why every
  figure can be redrawn on a laptop without cluster access.

The run's job ids, wall times and logs are in [verification.md](verification.md).